# ACB Experiments — Free Replication on Google Colab

This notebook runs the three experiments (P1, P2, P3) from the paper
*The Agent Coordination Bound (ACB)* using **free GPU on Colab + Ollama**.

**Requirements:** Colab with GPU runtime (Runtime → Change runtime type → T4 GPU).

---
## 1. Install Ollama

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils > /dev/null 2>&1
print('✓ zstd installed')

In [ ]:
!curl -fsSL https://ollama.ai/install.sh | sh
print('\n✓ Ollama installed')

In [ ]:
import subprocess, time, os
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/tmp/ollama_stdout.log', 'w'),
    stderr=open('/tmp/ollama_stderr.log', 'w'),
)
print(f'Ollama server started (PID: {proc.pid})')
time.sleep(10)
import urllib.request
try:
    resp = urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=5)
    print('✓ Ollama server is running')
except Exception as e:
    print(f'✗ Server not responding: {e}')
    !cat /tmp/ollama_stderr.log | tail -20

In [ ]:
MODEL = 'llama3.1:8b'
print(f'Pulling {MODEL}... (may take 5-10 min)')
!ollama pull {MODEL}
print(f'\n✓ {MODEL} ready')

In [ ]:
!ollama run {MODEL} 'What is 2+2? Answer with just the number.' --verbose 2>&1 | head -5
print('\n✓ Model is working')

---
## 2. Install ACB

In [ ]:
import os
os.chdir('/content')

# Clone repo (or use uploaded tar)
if not os.path.exists('/content/acb-experiments/acb'):
    # If you uploaded the tar.gz, extract it
    import glob
    tars = glob.glob('/content/*.tar.gz') + glob.glob('/content/drive/MyDrive/*.tar.gz')
    if tars:
        print(f'Extracting {tars[0]}...')
        !tar xzf {tars[0]} -C /content/
        # Handle double-nested folder: acb-experiments/acb-experiments/
        if os.path.exists('/content/acb-experiments/acb-experiments/acb'):
            !mv /content/acb-experiments/acb-experiments/* /content/acb-experiments/
            !rmdir /content/acb-experiments/acb-experiments 2>/dev/null || true
    else:
        print('Cloning from GitHub...')
        !git clone https://github.com/<YOUR-USERNAME>/acb-experiments.git

# Verify structure
assert os.path.exists('/content/acb-experiments/acb/__init__.py'), \
    'ERROR: acb/ not found. Check extraction path.'

os.chdir('/content/acb-experiments')
print(f'✓ Working directory: {os.getcwd()}')
!ls acb/ experiments/ monte_carlo/

In [ ]:
!pip install numpy scipy pandas matplotlib seaborn pyyaml httpx python-dotenv tqdm -q
print('✓ Dependencies installed')

In [ ]:
import os
os.environ['LOCAL_MODEL_URL'] = 'http://127.0.0.1:11434'
os.environ['LOCAL_MODEL_NAME'] = MODEL
os.environ['MAX_CONCURRENT'] = '1'
os.environ['SEED'] = '42'

with open('.env', 'w') as f:
    f.write(f'LOCAL_MODEL_URL=http://127.0.0.1:11434\n')
    f.write(f'LOCAL_MODEL_NAME={MODEL}\n')
    f.write(f'MAX_CONCURRENT=1\n')
    f.write(f'SEED=42\n')

print(f'✓ Backend: Ollama @ 127.0.0.1:11434 with {MODEL}')

In [ ]:
!PYTHONPATH=. python benchmarks/setup.py

---
## 3. Validate the Math (no GPU needed)

These run in seconds and confirm the analytical formulas.

In [ ]:
import sys
# Ensure we import from the right place
if '/content/acb-experiments' not in sys.path:
    sys.path.insert(0, '/content/acb-experiments')

# Monte Carlo validation — reproduces Table 3
from monte_carlo.validate_pharm import run_validation
results = run_validation(mc_runs=50000)

In [ ]:
# CBI diagnostic — reproduces Table 6
from acb.cbi import interpret_cbi

for name, n, a, c in [
    ('AutoGen GroupChat (3 agents)', 3, 0.72, 0.082),
    ('LangChain 5-agent workflow',   5, 0.72, 0.082),
    ('AgentPrune BEFORE pruning',   20, 0.51, 0.065),
    ('AgentPrune AFTER pruning',     8, 0.51, 0.065),
    ('Self-consistency k=40',       40, 0.56, 0.041),
]:
    print(f'{name}:\n  {interpret_cbi(n, a, c)}\n')

In [ ]:
# Greedy fleet selection — reproduces Table 7
from acb.greedy_fleet import greedy_fleet_select

agents = [
    ('Claude-3.5-Sonnet', 0.89), ('GPT-4o', 0.87),
    ('GPT-4o-inst2', 0.86), ('GPT-4-Turbo', 0.82),
    ('Gemini-1.5-Pro', 0.79), ('GPT-4o-mini', 0.72),
    ('LLaMA-3-70B', 0.57),
]
result = greedy_fleet_select(agents, c=0.082, cross_model_penalty=1.3)
print('Greedy Heterogeneous Fleet Selection')
print('=' * 65)
for s in result.steps:
    stop = ' ← STOP' if s.stop else ''
    print(f'  Step {s.step}: {s.agent_name:25s} ΔI={s.marginal_gain:+.4f}  I={s.cumulative_I:.4f}{stop}')
print(f'\nSelected: {len(result.selected)} agents → I(S*) = {result.total_I:.4f}')

In [ ]:
# Generate paper figures
from analysis.plots import generate_all_figures
generate_all_figures('figures/')

from IPython.display import Image, display
import glob
for fig in sorted(glob.glob('figures/*.png')):
    print(f'\n--- {os.path.basename(fig)} ---')
    display(Image(fig, width=600))

---
## 4. Run Live Experiments

Start with `--quick` (2 reps, 10 tasks) to verify, then run full.

In [ ]:
# QUICK TEST (~20-40 min)
!PYTHONPATH=/content/acb-experiments python run_all.py --quick --output-dir results/quick/

In [ ]:
import json, glob
for f in sorted(glob.glob('results/quick/*.json')):
    data = json.load(open(f))
    print(f'\n{"=" * 60}')
    print(f'{data.get("experiment_name", "?")}')
    print(f'{"=" * 60}')
    for k, v in data.get('summary', {}).items():
        if k != 'llm_usage': print(f'  {k}: {v}')

### Full Experiments (run individually, save to Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/acb-results

In [ ]:
# P1: Fleet sizing on HumanEval
!PYTHONPATH=/content/acb-experiments python -m experiments.p1_fleet_sizing \
    --fleet-sizes 1 2 3 4 5 6 7 8 9 10 12 15 --reps 50 --output-dir results/p1/
!cp -r results/p1/ /content/drive/MyDrive/acb-results/p1/
print('✓ P1 saved to Drive')

In [ ]:
# P2: Supervisor vs all-to-all on MATH
!PYTHONPATH=/content/acb-experiments python -m experiments.p2_topology_crossover \
    --fleet-sizes 1 2 3 4 5 6 7 8 9 10 12 15 --reps 50 --max-tasks 200 --output-dir results/p2/
!cp -r results/p2/ /content/drive/MyDrive/acb-results/p2/
print('✓ P2 saved to Drive')

In [ ]:
# P3: Shared vs isolated RAG
!PYTHONPATH=/content/acb-experiments python -m experiments.p3_rag_diversity \
    --fleet-sizes 1 3 6 9 --reps 50 --max-tasks 200 --output-dir results/p3/
!cp -r results/p3/ /content/drive/MyDrive/acb-results/p3/
print('✓ P3 saved to Drive')

---
## 5. Analyze Results

In [ ]:
import json, glob
for f in sorted(glob.glob('results/**/*.json', recursive=True) +
                glob.glob('/content/drive/MyDrive/acb-results/**/*.json', recursive=True)):
    data = json.load(open(f))
    s = data.get('summary', {})
    print(f'\n{"=" * 60}')
    print(f'{data.get("experiment_name", "?")}  ({os.path.basename(f)})')
    print(f'{"=" * 60}')
    for k, v in s.items():
        if k != 'llm_usage': print(f'  {k}: {v}')